#TA-1
####Name : Upadesh Gajanan Janjal
####Roll No : TY-AIDS-A-40
####Subject : ANNDL

# RNN Stock Price Prediction

This notebook walks you through building an AI model (Recurrent Neural Network) to predict stock prices using our `all_stocks_5yr.csv` dataset.
We will go through 5 main steps:
1. **Data Preparation**: Loading your dataset and formatting the stock data.
2. **Model Development**: Building the AI brain.
3. **Training**: Teaching the AI using historical data.
4. **Evaluation Metrics**: Checking how well the AI learned.
5. **Prediction & Visualization**: Forecasting future prices and plotting them on graphs.


In [ ]:
# Step 0: Install necessary libraries that Colab doesn't have by default
!pip install mplfinance joblib tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 3.8 MB/s eta 0:00:00


## Step 1: Data Preparation
First, we load your `all_stocks_5yr.csv` dataset. We will filter it to use Apple (AAPL) as an example.
We will also "normalize" (scale down) the data so our AI can process it much faster.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import joblib

# Set parameters
ticker = 'AAPL'
seq_length = 60 # How many past days the AI looks at to predict the next day

print("Loading the dataset from 'all_stocks_5yr.csv'...")
df = pd.read_csv('all_stocks_5yr.csv')

# Filter for the specific company
df = df[df['Name'] == ticker].copy()

# Convert dates to datetime objects and set as index
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)

# Rename columns to match standard conventions (capitalized)
df.rename(columns={'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close', 'volume': 'Volume'}, inplace=True)

# We use 5 features to help the AI learn better
features = ['Open', 'High', 'Low', 'Close', 'Volume']
data = df[features].copy()

# Fill any missing data points
data.ffill(inplace=True)
data.bfill(inplace=True)

# Scale data between 0 and 1 so the Neural Network can learn efficiently
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data)

# Create sequences (e.g., 60 days of data -> 1 next day target)
X, y = [], []
for i in range(seq_length, len(scaled_data)):
    X.append(scaled_data[i-seq_length:i])
    y.append(scaled_data[i, 3]) # The 3rd index is 'Close' price

X, y = np.array(X), np.array(y)

# Save the scaler for later use
joblib.dump(scaler, 'scaler.save')

print("Data Head:")
print(df.head())
print(f"\nData ready! We have {len(X)} samples of {seq_length} days each for {ticker}.")


Loading the dataset from 'all_stocks_5yr.csv'...
Data Head:
               Open     High      Low    Close     Volume  Name
date                                                           
2013-02-08  67.7142  68.4014  66.8928  67.8542  158168416  AAPL
2013-02-11  68.0714  69.2771  67.6071  68.5614  129029425  AAPL
2013-02-12  68.5014  68.9114  66.8205  66.8428  151829363  AAPL
2013-02-13  66.7442  67.6628  66.1742  66.7156  118721995  AAPL
2013-02-14  66.3599  67.3771  66.2885  66.6556   88809154  AAPL

Data ready! We have 1199 samples of 60 days each for AAPL.


## Step 2: Model Development
Now we build the AI model. We use an **LSTM (Long Short-Term Memory)** network.
LSTMs are a special type of RNN that are incredibly good at remembering long-term patterns, which is perfect for stock markets!


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# Build the Neural Network
model = Sequential()

# 1st LSTM Layer: Extracts patterns from our 60-day sequences
model.add(LSTM(units=50, return_sequences=True, input_shape=(X.shape[1], X.shape[2])))
model.add(Dropout(0.2)) # Randomly drops 20% of connections to prevent overfitting (memorization)

# 2nd LSTM Layer: Condenses the patterns
model.add(LSTM(units=50, return_sequences=False))
model.add(Dropout(0.2))

# Output Layer: Predicts the 1 final stock price
model.add(Dense(units=1))

print("AI Model Built! Here is the summary:")
model.summary()


AI Model Built! Here is the summary:


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 60, 50)         │        11,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 60, 50)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 50)             │        20,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 50)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31,451 (122.86 KB)

 Trainable params: 31,451 (122.86 KB)

 Non-trainable params: 0 (0.00 B)

## Step 3: Training the Model (with Real-Time TQDM Progress)
Now we teach the AI using our historical data. We split our data: 80% to train the AI, and 20% to test it.

**What happens during training?**
* **Epochs:** The model will go through the entire dataset 50 times (50 epochs) to learn the patterns.
* **Batch Size:** It processes 32 days of sequences at a time to update its internal weights efficiently.
* **Real-time TQDM Progress Bar:** We have integrated `tqdm` to show you a beautiful real-time progress bar in the cell output below. It will explicitly display the exact **time taken**, the **number of epochs**, the **steps per second**, and the **real-time loss values** as it trains!

We also use "callbacks" (EarlyStopping & ReduceLROnPlateau). These act as smart teachers, stopping the training early if the AI stops improving, and dynamically lowering the learning rate to fine-tune it towards the end.


In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tqdm.keras import TqdmCallback

# Split data into chronologically ordered Training (80%) and Testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# Compile the model using Adam optimizer and Mean Squared Error loss
optimizer = Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='mean_squared_error')

# Smart training assistants
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)

# TQDM real-time progress bar specifically optimized for Jupyter Notebooks/Colab
tqdm_callback = TqdmCallback(verbose=1)

print("Starting training process with TQDM progress tracking...")

# Notice we set verbose=0 in model.fit so TQDM takes full control over the visual output
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=0,
    callbacks=[early_stopping, reduce_lr, tqdm_callback]
)

# Save the trained model
model.save('lstm_model.keras')
print("\nModel trained successfully! Saved as 'lstm_model.keras'")


0epoch [00:00, ?epoch/s]

0batch [00:00, ?batch/s]

Starting training process with TQDM progress tracking...

Epoch 8: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 17: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.

Epoch 26: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.

Epoch 31: ReduceLROnPlateau reducing learning rate to 6.25000029685907e-05.
Epoch 33: early stopping
Restoring model weights from the end of the best epoch: 23.

Model trained successfully! Saved as 'lstm_model.keras'


## Step 4: Evaluation Metrics
Let's see how well our model did on the 20% test data it has never seen before.
We calculate MAE (Average dollar mistake) and MAPE (Average percentage mistake).


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

print("Evaluating model performance on test data...")
y_pred_scaled = model.predict(X_test, verbose=0)

# The model outputs scaled numbers (between 0 and 1). We need to convert them back to actual dollar prices.
dummy_pred = np.zeros((len(y_pred_scaled), 5))
dummy_pred[:, 3] = y_pred_scaled[:, 0]
y_pred = scaler.inverse_transform(dummy_pred)[:, 3]

dummy_actual = np.zeros((len(y_test), 5))
dummy_actual[:, 3] = y_test
y_actual = scaler.inverse_transform(dummy_actual)[:, 3]

# Calculate errors
mae = mean_absolute_error(y_actual, y_pred)
rmse = np.sqrt(mean_squared_error(y_actual, y_pred))
mape = mean_absolute_percentage_error(y_actual, y_pred) * 100

print(f"Mean Absolute Error (MAE): ${mae:.2f}")
print(f"Root Mean Squared Error (RMSE): ${rmse:.2f}")
print(f"Mean Absolute Percentage Error (MAPE): {mape:.2f}%")


## Step 5: Prediction & Visualization
Now for the fun part! We grab the last 60 days of data from your CSV file, feed it to our AI, and ask it to predict the NEXT day's stock price.
Then, we plot a Candlestick chart to visualize the recent market trends.


In [ ]:
import matplotlib.pyplot as plt
import mplfinance as mpf
from datetime import datetime, timedelta

# 1. Grab the last 60 days of data from your dataset
df_live = data.tail(seq_length).copy()

# 2. Scale the data and predict
scaled_live = scaler.transform(df_live)
X_input = scaled_live.reshape(1, seq_length, len(features))

predicted_scaled = model.predict(X_input, verbose=0)

# 3. Convert prediction back to dollar value
dummy_array = np.zeros((1, len(features)))
dummy_array[0, 3] = predicted_scaled[0][0]
predicted_price = scaler.inverse_transform(dummy_array)[0, 3]

last_actual_price = df_live['Close'].iloc[-1]
last_date = df_live.index[-1].strftime('%Y-%m-%d')

print("-" * 40)
print(f"Latest Dataset Date: {last_date}")
print(f"Last Actual Close Price: ${last_actual_price:.2f}")
print(f"Forecasted Next Day Close Price: ${predicted_price:.2f}")
print("-" * 40)

# 4. Plotting
print("Generating Candlestick chart with recent historical data...")
mpf.plot(df_live, type='candle', style='charles',
         title=f'{ticker} Recent Trend from Dataset',
         ylabel='Price ($)')


####Github link : https://github.com/upadesh94/ANNDL/blob/main/TA